In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# This cell verifies the project paths and confirms the AMI resources available for the base-paper reproduction.

import os
import json
import glob

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

print("=" * 70)
print("BASE-PAPER REPRODUCTION — AMI RESOURCE CHECK")
print("=" * 70)

print("Project directory:", PROJECT_DIR)
print("AMI directory:", AMI_DIR)
print("AMI exists:", os.path.exists(AMI_DIR))

meeting_id = "ES2004a"
meeting_dir = os.path.join(AMI_DIR, meeting_id)

print("\nMeeting:", meeting_id)
print("Meeting directory:", meeting_dir)
print("Exists:", os.path.exists(meeting_dir))

if os.path.exists(meeting_dir):

    files = glob.glob(
        os.path.join(meeting_dir, "**", "*"),
        recursive=True
    )

    files = [
        f for f in files
        if os.path.isfile(f)
    ]

    print("\nFiles available:")

    for f in files:
        print(" -", f)

BASE-PAPER REPRODUCTION — AMI RESOURCE CHECK
Project directory: /content/drive/MyDrive/MTechIndProj/MoM_Project
AMI directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami
AMI exists: True

Meeting: ES2004a
Meeting directory: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a
Exists: True

Files available:
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_reference_summary.txt
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/ES2004a_verification.json
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/video/ES2004a.PreferredOverview.avi
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/video/ES2004a.Closeup1.avi
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
 - /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/slides/ES2004a.586.44__693.07.jpg
 - /content/drive/MyDrive/MTechIndProj/MoM_

In [ ]:
# This cell installs the Whisper implementation and audio dependencies required for reproducing the ASR stage.

!pip install -q openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# This cell verifies that Whisper can access the Tesla T4 GPU before processing the meeting audio.

import torch
import whisper

print("=" * 70)
print("WHISPER ENVIRONMENT CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Whisper imported successfully.")

WHISPER ENVIRONMENT CHECK
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
Whisper imported successfully.


In [ ]:
# This cell defines the ES2004a AMI audio file that will be transcribed using Whisper.

AUDIO_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami",
    "ES2004a",
    "audio",
    "ES2004a.Mix-Headset.wav"
)

print("=" * 70)
print("WHISPER INPUT")
print("=" * 70)

print("Audio path:", AUDIO_PATH)
print("Exists:", os.path.exists(AUDIO_PATH))

if os.path.exists(AUDIO_PATH):
    size_mb = os.path.getsize(AUDIO_PATH) / (1024 * 1024)
    print(f"Audio size: {size_mb:.2f} MB")

WHISPER INPUT
Audio path: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami/ES2004a/audio/ES2004a.Mix-Headset.wav
Exists: True
Audio size: 32.02 MB


In [ ]:
# This cell loads the Whisper-small model on the available GPU for the initial AMI ASR reproduction test.

WHISPER_MODEL_NAME = "small"

whisper_model = whisper.load_model(
    WHISPER_MODEL_NAME,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

print("=" * 70)
print("WHISPER MODEL LOADED")
print("=" * 70)

print("Model:", WHISPER_MODEL_NAME)
print("Device:", whisper_model.device)

100%|████████████████████████████████████████| 461M/461M [00:03<00:00, 149MiB/s]


WHISPER MODEL LOADED
Model: small
Device: cuda:0


In [ ]:
# This cell runs Whisper on the ES2004a audio and produces a timestamped ASR transcription.

print("=" * 70)
print("WHISPER ASR — ES2004a")
print("=" * 70)

whisper_result = whisper_model.transcribe(
    AUDIO_PATH,
    language="en",
    fp16=torch.cuda.is_available(),
    verbose=False
)

print("Transcription completed.")

print(
    "Number of segments:",
    len(whisper_result["segments"])
)

print("\nFirst 10 Whisper segments:\n")

for segment in whisper_result["segments"][:10]:

    print(
        f"[{segment['start']:.2f} - "
        f"{segment['end']:.2f}] "
        f"{segment['text'].strip()}"
    )

WHISPER ASR — ES2004a


100%|██████████| 104935/104935 [01:06<00:00, 1571.42frames/s]

Transcription completed.
Number of segments: 327

First 10 Whisper segments:

[0.00 - 16.00] We're not allowed to dim lights so we can see that a little better.
[16.00 - 17.00] Yeah.
[17.00 - 18.00] Okay.
[18.00 - 19.00] That's fine.
[19.00 - 24.00] Am I supposed to be standing up there?
[24.00 - 27.00] So we've got both of these clipped on.
[28.00 - 30.00] She can own some people.
[30.00 - 31.00] Yeah, it's good.
[31.00 - 32.00] Both of them.
[32.00 - 33.00] Okay.


In [ ]:
# This cell saves the Whisper ASR output for ES2004a without modifying the original AMI transcript.

import json

ASR_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

os.makedirs(ASR_OUTPUT_DIR, exist_ok=True)

ASR_OUTPUT_PATH = os.path.join(
    ASR_OUTPUT_DIR,
    "ES2004a_whisper_small.json"
)

with open(
    ASR_OUTPUT_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "meeting_id": "ES2004a",
            "model": "openai-whisper-small",
            "language": "en",
            "segments": whisper_result["segments"],
            "text": whisper_result["text"]
        },
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 70)
print("WHISPER ASR SAVED")
print("=" * 70)

print("Path:", ASR_OUTPUT_PATH)
print("Exists:", os.path.exists(ASR_OUTPUT_PATH))

WHISPER ASR SAVED
Path: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_whisper_small.json
Exists: True


In [ ]:
# This cell prepares the Word Error Rate metric for evaluating our Whisper transcription against the AMI transcript.

import importlib.util

if importlib.util.find_spec("jiwer") is None:
    !pip install -q jiwer

from jiwer import wer

print("=" * 70)
print("WER EVALUATION READY")
print("=" * 70)

print("jiwer imported successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.9 MB/s eta 0:00:00
WER EVALUATION READY
jiwer imported successfully.


In [ ]:
# This cell calculates the Word Error Rate of Whisper-small against the original AMI ES2004a transcript.

AMI_TRANSCRIPT_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami",
    "ES2004a",
    "transcript",
    "ES2004a_transcript.txt"
)

with open(
    AMI_TRANSCRIPT_PATH,
    "r",
    encoding="utf-8"
) as f:
    ami_transcript = f.read()

whisper_transcript = whisper_result["text"]

es2004a_wer = wer(
    ami_transcript,
    whisper_transcript
)

print("=" * 70)
print("WHISPER ASR — WER RESULT")
print("=" * 70)

print("Meeting: ES2004a")
print("Model: openai-whisper-small")
print(f"WER: {es2004a_wer:.4f}")
print(f"WER (%): {es2004a_wer * 100:.2f}%")

WHISPER ASR — WER RESULT
Meeting: ES2004a
Model: openai-whisper-small
WER: 1.0000
WER (%): 100.00%


In [ ]:
# This cell removes AMI timestamps, speaker labels, punctuation, and formatting differences before calculating WER.

import re
from jiwer import wer

def normalize_ami_transcript(text):
    """
    Convert AMI timestamped/annotated transcript into plain reference text.
    """

    # Remove timestamp prefixes such as [10.99 - 11.02]
    text = re.sub(
        r"\[\s*\d+(?:\.\d+)?\s*-\s*\d+(?:\.\d+)?\s*\]",
        " ",
        text
    )

    # Remove speaker labels such as A:, B:, C:, D:
    text = re.sub(
        r"\b[A-D]\s*:",
        " ",
        text
    )

    # Convert to lowercase
    text = text.lower()

    # Keep words/numbers and remove punctuation
    text = re.sub(
        r"[^a-z0-9\s']",
        " ",
        text
    )

    # Normalize whitespace
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# Normalize AMI reference
ami_reference_clean = normalize_ami_transcript(
    ami_transcript
)

# Normalize Whisper output using the same procedure
whisper_reference_clean = normalize_ami_transcript(
    whisper_transcript
)

# Calculate WER
es2004a_wer_clean = wer(
    ami_reference_clean,
    whisper_reference_clean
)

print("=" * 70)
print("CORRECTED WHISPER ASR — WER")
print("=" * 70)

print("Meeting:", "ES2004a")
print("Model:", "openai-whisper-small")

print(
    f"WER: {es2004a_wer_clean:.4f}"
)

print(
    f"WER (%): {es2004a_wer_clean * 100:.2f}%"
)

print("\nReference words:",
      len(ami_reference_clean.split()))

print("Whisper words:",
      len(whisper_reference_clean.split()))

CORRECTED WHISPER ASR — WER
Meeting: ES2004a
Model: openai-whisper-small
WER: 0.2914
WER (%): 29.14%

Reference words: 2653
Whisper words: 2122


In [ ]:
# This cell displays the beginning of the normalized AMI reference and Whisper transcription for a sanity check.

print("=" * 70)
print("NORMALIZED TRANSCRIPT COMPARISON")
print("=" * 70)

print("\nAMI REFERENCE — first 500 characters:")
print(ami_reference_clean[:500])

print("\nWHISPER — first 500 characters:")
print(whisper_reference_clean[:500])

NORMALIZED TRANSCRIPT COMPARISON

AMI REFERENCE — first 500 characters:
hmm hmm hmm are we we're not allowed to dim the lights so people can see that a bit better yeah okay that's fine am i supposed to be standing up there so okay we've got both of these clipped on she gonna answer me yeah or not i've got right both of them okay yes god jesus it's gonna fall off okay yep yep okay okay tu tu tu tu hello everybody hi good morning um i'm sarah the project manager and this is our first meeting surprisingly enough okay this is our agenda um we will do some stuff get to k

WHISPER — first 500 characters:
we're not allowed to dim lights so we can see that a little better yeah okay that's fine am i supposed to be standing up there so we've got both of these clipped on she can own some people yeah it's good both of them okay yeah good i'm just going to fall off it doesn't it okay hello everybody i'm sarah project manager and this is our first meeting surprisingly nice okay this is our agenda we

In [ ]:
# This cell runs Whisper-small on all 10 AMI meetings and saves each timestamped transcription separately.

MEETING_IDS = [
    "ES2004a",
    "ES2004b",
    "ES2004c",
    "ES2004d",
    "ES2005a",
    "ES2005b",
    "ES2005c",
    "ES2006a",
    "ES2006b",
    "ES2008a"
]

ASR_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

os.makedirs(ASR_OUTPUT_DIR, exist_ok=True)

print("=" * 70)
print("WHISPER ASR — ALL AMI MEETINGS")
print("=" * 70)

asr_results = {}

for index, meeting_id in enumerate(MEETING_IDS, start=1):

    print(
        f"\n[{index}/{len(MEETING_IDS)}] "
        f"Processing {meeting_id}..."
    )

    audio_path = os.path.join(
        PROJECT_DIR,
        "data",
        "raw",
        "ami",
        meeting_id,
        "audio",
        f"{meeting_id}.Mix-Headset.wav"
    )

    output_path = os.path.join(
        ASR_OUTPUT_DIR,
        f"{meeting_id}_whisper_small.json"
    )

    if not os.path.exists(audio_path):
        print("⚠ Audio missing:", audio_path)
        continue

    result = whisper_model.transcribe(
        audio_path,
        language="en",
        fp16=torch.cuda.is_available(),
        verbose=False
    )

    asr_results[meeting_id] = result

    with open(
        output_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            {
                "meeting_id": meeting_id,
                "model": "openai-whisper-small",
                "language": "en",
                "segments": result["segments"],
                "text": result["text"]
            },
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        "  Segments:",
        len(result["segments"])
    )

    print(
        "  Saved:",
        output_path
    )

print("\n" + "=" * 70)
print("WHISPER ASR RUN COMPLETE")
print("=" * 70)

print(
    "Meetings processed:",
    len(asr_results)
)

WHISPER ASR — ALL AMI MEETINGS

[1/10] Processing ES2004a...


100%|██████████| 104935/104935 [01:00<00:00, 1723.92frames/s]


  Segments: 350
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004a_whisper_small.json

[2/10] Processing ES2004b...


100%|██████████| 234549/234549 [02:28<00:00, 1584.56frames/s]


  Segments: 1017
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2004b_whisper_small.json

[3/10] Processing ES2004c...


...

  Segments: 165
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2006a_whisper_small.json

[9/10] Processing ES2006b...


100%|██████████| 218312/218312 [02:06<00:00, 1726.02frames/s]


  Segments: 535
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2006b_whisper_small.json

[10/10] Processing ES2008a...


 97%|█████████▋| 101336/104336 [01:00<00:01, 1686.68frames/s]

  Segments: 236
  Saved: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts/ES2008a_whisper_small.json

WHISPER ASR RUN COMPLETE
Meetings processed: 10


In [ ]:
# This cell calculates normalized Word Error Rate for Whisper-small across all 10 AMI meetings.

import pandas as pd
import json
import os

wer_results = []

for meeting_id in MEETING_IDS:

    print(f"Evaluating {meeting_id}...")

    # AMI reference transcript
    ami_path = os.path.join(
        PROJECT_DIR,
        "data",
        "raw",
        "ami",
        meeting_id,
        "transcript",
        f"{meeting_id}_transcript.txt"
    )

    # Our Whisper output
    whisper_path = os.path.join(
        PROJECT_DIR,
        "data",
        "transcripts",
        f"{meeting_id}_whisper_small.json"
    )

    if not os.path.exists(ami_path):
        print("  ⚠ AMI transcript missing")
        continue

    if not os.path.exists(whisper_path):
        print("  ⚠ Whisper output missing")
        continue

    with open(ami_path, "r", encoding="utf-8") as f:
        ami_text = f.read()

    with open(whisper_path, "r", encoding="utf-8") as f:
        whisper_data = json.load(f)

    whisper_text = whisper_data["text"]

    # Normalize both transcripts
    reference = normalize_ami_transcript(ami_text)
    hypothesis = normalize_ami_transcript(whisper_text)

    meeting_wer = wer(
        reference,
        hypothesis
    )

    wer_results.append({
        "meeting_id": meeting_id,
        "reference_words": len(reference.split()),
        "whisper_words": len(hypothesis.split()),
        "wer": meeting_wer,
        "wer_percent": meeting_wer * 100
    })

    print(
        f"  WER: {meeting_wer:.4f} "
        f"({meeting_wer * 100:.2f}%)"
    )


wer_df = pd.DataFrame(wer_results)

print("\n" + "=" * 70)
print("WHISPER ASR — ALL MEETINGS WER")
print("=" * 70)

display(wer_df)

print("\nAverage WER:",
      f"{wer_df['wer'].mean():.4f}")

print("Average WER (%):",
      f"{wer_df['wer_percent'].mean():.2f}%")

Evaluating ES2004a...
  WER: 0.3072 (30.72%)
Evaluating ES2004b...
  WER: 0.2834 (28.34%)
Evaluating ES2004c...
  WER: 0.2482 (24.82%)
Evaluating ES2004d...
  WER: 0.3290 (32.90%)
Evaluating ES2005a...
  WER: 0.3835 (38.35%)
Evaluating ES2005b...
  WER: 0.3194 (31.94%)
Evaluating ES2005c...
  WER: 0.3234 (32.34%)
Evaluating ES2006a...
  WER: 0.2099 (20.99%)
Evaluating ES2006b...
  WER: 0.2624 (26.24%)
Evaluating ES2008a...
  WER: 0.2778 (27.78%)

WHISPER ASR — ALL MEETINGS WER


,meeting_id,reference_words,whisper_words,wer,wer_percent
0,ES2004a,2653,2131,0.307199,30.719940
1,ES2004b,6874,5587,0.283387,28.338667
2,ES2004c,7084,5835,0.248165,24.816488
3,ES2004d,6204,4982,0.328981,32.898130
4,ES2005a,764,694,0.383508,38.350785
5,ES2005b,6297,4934,0.319358,31.935842
6,ES2005c,6802,5340,0.323434,32.343428
7,ES2006a,2806,2326,0.209907,20.990734
8,ES2006b,6277,5057,0.262386,26.238649
9,ES2008a,2541,2161,0.277843,27.784337



Average WER: 0.2944
Average WER (%): 29.44%


In [ ]:
# This cell saves the Whisper WER evaluation so it can be used later in the thesis and final comparison.

WER_OUTPUT_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

os.makedirs(
    WER_OUTPUT_DIR,
    exist_ok=True
)

WER_OUTPUT_PATH = os.path.join(
    WER_OUTPUT_DIR,
    "whisper_small_wer.csv"
)

wer_df.to_csv(
    WER_OUTPUT_PATH,
    index=False
)

print("=" * 70)
print("WHISPER WER RESULTS SAVED")
print("=" * 70)

print(WER_OUTPUT_PATH)

WHISPER WER RESULTS SAVED
/content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results/whisper_small_wer.csv


In [ ]:
# This cell checks the installed pyannote.audio version and confirms whether the diarization package is available.

import importlib.util

print("=" * 70)
print("PYANNOTE DIARIZATION ENVIRONMENT CHECK")
print("=" * 70)

pyannote_available = (
    importlib.util.find_spec("pyannote.audio") is not None
)

print("pyannote.audio available:", pyannote_available)

if pyannote_available:
    import pyannote.audio
    print("pyannote.audio version:", pyannote.audio.__version__)
else:
    print("pyannote.audio is not installed.")

PYANNOTE DIARIZATION ENVIRONMENT CHECK


ModuleNotFoundError: No module named 'pyannote'

In [ ]:
# This cell installs pyannote.audio 3.1 for the speaker diarization stage of the project.

!pip install -q "pyannote.audio==3.1.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.7/208.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/

In [ ]:
# This cell checks the current NumPy, Numba, PyTorch, and pyannote versions before fixing the dependency conflict.

import numpy
import torch

print("=" * 70)
print("ENVIRONMENT VERSION CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)

try:
    import numba
    print("Numba:", numba.__version__)
except Exception as e:
    print("Numba import error:", e)

try:
    import pyannote.audio
    print("pyannote.audio:", pyannote.audio.__version__)
except Exception as e:
    print("pyannote import error:", e)

ENVIRONMENT VERSION CHECK
NumPy: 2.1.3
PyTorch: 2.11.0+cu128
Numba: 0.61.2
pyannote import error: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'


In [ ]:
# This cell checks the scientific Python packages involved in the pyannote import error before making any environment changes.

import numpy
import scipy
import sklearn
import torch

print("=" * 70)
print("PYANNOTE DEPENDENCY DIAGNOSTICS")
print("=" * 70)

print("Python:", __import__("sys").version)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)
print("PyTorch:", torch.__version__)

try:
    import torchaudio
    print("Torchaudio:", torchaudio.__version__)
except Exception as e:
    print("Torchaudio ERROR:", repr(e))

try:
    import librosa
    print("Librosa:", librosa.__version__)
except Exception as e:
    print("Librosa ERROR:", repr(e))

try:
    import pyannote.core
    print("pyannote.core:", pyannote.core.__version__)
except Exception as e:
    print("pyannote.core ERROR:", repr(e))

AttributeError: module 'numpy._core._multiarray_umath' has no attribute '_blas_supports_fpe'

In [ ]:
# This cell repairs the NumPy, SciPy, and scikit-learn compatibility stack required by pyannote.

!pip install -q --force-reinstall --no-cache-dir \
    "numpy==2.2.6" \
    "scipy==1.15.3" \
    "scikit-learn==1.7.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 329.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 297.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 321.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 351.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 309.1/309.1 kB 428.5 MB/s eta 0:00:00


In [ ]:
# This cell verifies that NumPy, SciPy, and scikit-learn can import correctly after the repair.

import numpy
import scipy
import sklearn

print("=" * 70)
print("REPAIRED SCIENTIFIC STACK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("Scikit-learn:", sklearn.__version__)

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [ ]:
# This cell checks the fresh Colab runtime before installing any additional diarization dependencies.

import sys
import numpy
import torch

print("=" * 70)
print("FRESH COLAB ENVIRONMENT")
print("=" * 70)

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

FRESH COLAB ENVIRONMENT
Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy: 2.2.6
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# This cell restores the project directory and AMI paths after restarting the Colab runtime.

import os
import torch

PROJECT_DIR = "/content/drive/MyDrive/MTechIndProj/MoM_Project"

AMI_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "raw",
    "ami"
)

TRANSCRIPT_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "transcripts"
)

EVALUATION_DIR = os.path.join(
    PROJECT_DIR,
    "evaluation_results"
)

print("=" * 70)
print("PROJECT ENVIRONMENT RESTORED")
print("=" * 70)

print("Project:", PROJECT_DIR)
print("AMI:", AMI_DIR)
print("Transcripts:", TRANSCRIPT_DIR)
print("Evaluation:", EVALUATION_DIR)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PROJECT ENVIRONMENT RESTORED
Project: /content/drive/MyDrive/MTechIndProj/MoM_Project
AMI: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/raw/ami
Transcripts: /content/drive/MyDrive/MTechIndProj/MoM_Project/data/transcripts
Evaluation: /content/drive/MyDrive/MTechIndProj/MoM_Project/evaluation_results

CUDA available: True
GPU: Tesla T4


In [ ]:
# This cell checks whether pyannote.audio is available in the fresh runtime without modifying the environment.

import importlib.util

pyannote_spec = importlib.util.find_spec("pyannote")

print("=" * 70)
print("PYANNOTE AVAILABILITY")
print("=" * 70)

print("pyannote package available:", pyannote_spec is not None)

if pyannote_spec is not None:
    try:
        import pyannote.audio
        print("pyannote.audio version:", pyannote.audio.__version__)
    except Exception as e:
        print("pyannote.audio import error:", repr(e))
else:
    print("pyannote is not installed.")

PYANNOTE AVAILABILITY
pyannote package available: True
pyannote.audio import error: AttributeError("module 'torchaudio' has no attribute 'set_audio_backend'")


In [ ]:
# This cell installs the current pyannote.audio package for speaker diarization.

!pip install -q pyannote.audio

In [ ]:
# This cell verifies that pyannote.audio imports correctly in the clean runtime.

import pyannote.audio

print("=" * 70)
print("PYANNOTE INSTALLATION CHECK")
print("=" * 70)

print("pyannote.audio version:", pyannote.audio.__version__)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

AttributeError: module 'torchaudio' has no attribute 'set_audio_backend'

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE / TORCHAUDIO VERSIONS")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE / TORCHAUDIO VERSIONS
pyannote.audio: 3.1.1
torchaudio: 2.11.0+cu128
torch: 2.11.0+cu128


In [ ]:
# Remove the incompatible pyannote.audio 3.1.1
!pip uninstall -y pyannote.audio

# Install the current pyannote.audio
!pip install -q -U pyannote.audio

Found existing installation: pyannote.audio 3.1.1
Uninstalling pyannote.audio-3.1.1:
  Successfully uninstalled pyannote.audio-3.1.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 20.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.44.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.44.0 which is incompatible.


In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE VERSION AFTER UPDATE")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE VERSION AFTER UPDATE
pyannote.audio: 4.0.7
torchaudio: 2.11.0+cu128
torch: 2.11.0+cu128


In [ ]:
import torch
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT TEST")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

print("PyTorch:",
      torch.__version__)

print("CUDA:",
      torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:",
          torch.cuda.get_device_name(0))

PYANNOTE IMPORT TEST
pyannote.audio: 4.0.7
PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


If you already have a Hugging Face token, you can log in through Colab's secret mechanism rather than putting the token directly in the notebook.

In Colab:

Secrets → add HF_TOKEN

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("=" * 70)
print("HUGGING FACE TOKEN CHECK")
print("=" * 70)

if HF_TOKEN:
    print("HF_TOKEN found: True")
    print("Token length:", len(HF_TOKEN))
else:
    print("HF_TOKEN found: False")

HUGGING FACE TOKEN CHECK
HF_TOKEN found: True
Token length: 37


In [ ]:
from pyannote.audio import Pipeline
import torch

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

pipeline.to(torch.device("cuda"))

print("Diarization pipeline loaded successfully.")
print("Device: CUDA")

ModuleNotFoundError: No module named 'pyannote'

In [ ]:
!pip install -q pyannote.audio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE INSTALLATION")
print("=" * 70)

print("pyannote.audio:",
      metadata.version("pyannote.audio"))

print("torchaudio:",
      metadata.version("torchaudio"))

print("torch:",
      metadata.version("torch"))

PYANNOTE INSTALLATION
pyannote.audio: 4.0.7
torchaudio: 2.11.0+cpu
torch: 2.11.0+cpu


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT TEST")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [ ]:
!pip install -q --force-reinstall \
    torch==2.11.0 \
    torchaudio==2.11.0 \
    --index-url https://download.pytorch.org/whl/cu128

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 22.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 66.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 74.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 35.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 156.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 89.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 171.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 156.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 93.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 55.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 55.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import torch
import torchaudio
import numpy

print("=" * 70)
print("POST-REPAIR ENVIRONMENT CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("PyTorch:", torch.__version__)
print("TorchAudio:", torchaudio.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

POST-REPAIR ENVIRONMENT CHECK
NumPy: 2.5.2
PyTorch: 2.11.0+cu128
TorchAudio: 2.11.0+cu128
CUDA available: False


In [ ]:
import torch

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU CHECK
PyTorch: 2.11.0+cpu
CUDA: False
CUDA version: None
GPU count: 0


In [ ]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import os
import torch
import subprocess

print("=" * 70)
print("COLAB GPU DIAGNOSTIC")
print("=" * 70)

print("CUDA_VISIBLE_DEVICES:",
      os.environ.get("CUDA_VISIBLE_DEVICES"))

print("NVIDIA_VISIBLE_DEVICES:",
      os.environ.get("NVIDIA_VISIBLE_DEVICES"))

print("PyTorch:", torch.__version__)
print("PyTorch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

print("\nChecking NVIDIA driver...")

result = subprocess.run(
    ["bash", "-c", "command -v nvidia-smi || true"],
    capture_output=True,
    text=True
)

print("nvidia-smi location:", result.stdout.strip() or "NOT FOUND")

COLAB GPU DIAGNOSTIC
CUDA_VISIBLE_DEVICES: None
NVIDIA_VISIBLE_DEVICES: all
PyTorch: 2.11.0+cu128
PyTorch CUDA: 12.8
CUDA available: True
GPU count: 1

Checking NVIDIA driver...
nvidia-smi location: /opt/bin/nvidia-smi


In [ ]:
import torch

print("=" * 70)
print("GPU CONFIRMATION")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU CONFIRMATION
CUDA available: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE CHECK")
print("=" * 70)

print("pyannote.audio:",
      pyannote.audio.__version__)

ModuleNotFoundError: No module named 'pyannote'

In [ ]:
!pip install -q "pyannote.audio==4.0.7"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 61.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.1/52.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983

In [ ]:
import importlib.metadata as metadata

print("=" * 70)
print("PYANNOTE INSTALLATION CHECK")
print("=" * 70)

print("pyannote.audio:", metadata.version("pyannote.audio"))
print("torch:", metadata.version("torch"))
print("torchaudio:", metadata.version("torchaudio"))

PYANNOTE INSTALLATION CHECK
pyannote.audio: 4.0.7
torch: 2.11.0+cu128
torchaudio: 2.11.0+cu128


In [ ]:
from pyannote.audio import Pipeline
import torch

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

# Retrieve the Hugging Face token from Colab Secrets
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

# Use GPU
pipeline.to(torch.device("cuda"))

print("Diarization pipeline loaded successfully.")
print("Device: CUDA")

ImportError: cannot import name '_slice' from 'numpy._core.umath' (/usr/local/lib/python3.13/dist-packages/numpy/_core/umath.py)

In [ ]:
!pip install -q --force-reinstall "numpy==2.2.6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 106.4 MB/s eta 0:00:00


In [ ]:
import numpy
import scipy

print("=" * 70)
print("NUMPY / SCIPY CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)

NUMPY / SCIPY CHECK
NumPy: 2.1.3
SciPy: 1.16.3


In [ ]:
import torch

print("=" * 70)
print("CUDA CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA CHECK
PyTorch: 2.11.0+cu128
CUDA: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT CHECK")
print("=" * 70)

print("pyannote.audio:", pyannote.audio.__version__)

AttributeError: 'numpy.ufunc' object has no attribute '__module__' and no __dict__ for setting new attributes

In [ ]:
import sys
import numpy

print("Python:", sys.version)
print("NumPy:", numpy.__version__)

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
NumPy: 2.1.3


In [ ]:
!pip install -q --force-reinstall \
    "numpy==2.2.6" \
    "scipy==1.15.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 25.6 MB/s eta 0:00:00


In [ ]:
import numpy
import scipy

print("=" * 70)
print("SCIENTIFIC STACK CHECK")
print("=" * 70)

print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)

SCIENTIFIC STACK CHECK
NumPy: 2.2.6
SciPy: 1.15.3


In [ ]:
import torch

print("=" * 70)
print("PYTORCH / GPU CHECK")
print("=" * 70)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PYTORCH / GPU CHECK
PyTorch: 2.11.0+cu128
CUDA: True
GPU count: 1
GPU: Tesla T4


In [ ]:
import scipy.signal
import torchmetrics
import lightning

print("=" * 70)
print("DEPENDENCY CHAIN CHECK")
print("=" * 70)

print("SciPy: OK")
print("TorchMetrics: OK")
print("Lightning: OK")

DEPENDENCY CHAIN CHECK
SciPy: OK
TorchMetrics: OK
Lightning: OK


In [ ]:
import pyannote.audio

print("=" * 70)
print("PYANNOTE IMPORT CHECK")
print("=" * 70)

print("pyannote.audio:", pyannote.audio.__version__)

PYANNOTE IMPORT CHECK
pyannote.audio: 4.0.7


In [ ]:
import torch
from pyannote.audio import Pipeline
from google.colab import userdata

print("=" * 70)
print("LOADING PYANNOTE DIARIZATION PIPELINE")
print("=" * 70)

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN available:", bool(HF_TOKEN))
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN
)

pipeline.to(torch.device("cuda"))

print("\nDiarization pipeline loaded successfully.")
print("Device: CUDA")

LOADING PYANNOTE DIARIZATION PIPELINE
HF_TOKEN available: True
CUDA available: True
GPU: Tesla T4


config.yaml:   0%|          | 0.00/444 [00:00<?, ?B/s]

segmentation/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 5.91MB            

segmentation/pytorch_model.bin: downloading bytes:           |  0.00B            

plda/xvec_transform.npz: reconstructing file:   0%|          |  0.00B /  134kB            

...

plda/plda.npz: reconstructing file:   0%|          |  0.00B /  134kB            

plda/plda.npz: downloading bytes:           |  0.00B            

embedding/pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 26.6MB            

embedding/pytorch_model.bin: downloading bytes:           |  0.00B            


Diarization pipeline loaded successfully.
Device: CUDA


In [ ]:
from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive
